In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Temizlenmiş "Tek Gerçek Kaynak" Verimizi Okuyalım
df = pd.read_csv("../data/processed/01_final_merged_data.csv", sep=';', decimal=',')

# ==========================================
# 2. ÇOK SINIFLI HEDEF (MULTI-CLASS TARGET) OLUŞTURMA
# ==========================================
def target_belirle(row):
    surec = str(row.get('Süreç', '')).strip()
    neden = str(row.get('İade Nedeni', '')).strip()
    
    # Kural 1: Süreç İptal ise -> 2
    if surec == 'İptal':
        return 2
    # Kural 2: Süreç İade VEYA İade Nedeni doluysa (Süreç teslim görünse bile) -> 1
    elif surec == 'İade' or (neden != 'nan' and neden != ''):
        return 1
    # Kural 3: Geri kalan her şey başarılı teslimattır -> 0
    else:
        return 0

# apply fonksiyonu ile her satır için target sütunumuzu üretiyoruz
df['target'] = df.apply(target_belirle, axis=1)

print("--- Hedef Değişken (0: Teslim, 1: İade, 2: İptal) Dağılımı ---")
print(df['target'].value_counts(normalize=True) * 100)
print("\n")


# ==========================================
# 3. KARDİNALİTE DÜŞÜRME (BOYUT İNDİRGEME)
# ==========================================
# 81 ili modele sokmak yerine, siparişlerin en yoğun olduğu ilk 10 ili tutuyoruz. 
# Geri kalan 71 ili 'Diğer' torbasına atarak modeli gereksiz gürültüden (noise) kurtarıyoruz.
top_10_iller = df['İl (Teslimat)'].value_counts().nlargest(10).index
df['İl_Gruplu'] = df['İl (Teslimat)'].where(df['İl (Teslimat)'].isin(top_10_iller), 'Diğer')

# Aynı stratejiyi Alt Kategori için de uygulayalım (En çok hareket gören ilk 10 alt kategori)
top_10_altkat = df['Alt Kategori'].value_counts().nlargest(10).index
df['AltKat_Gruplu'] = df['Alt Kategori'].where(df['Alt Kategori'].isin(top_10_altkat), 'Diğer')

print("--- İl Dağılımı (Gruplama Sonrası) ---")
print(df['İl_Gruplu'].value_counts().head(5))
print("\n")

def platform_birlestir(x):
    x_lower = str(x).lower()
    # İçinde admin geçen tüm varyasyonları tek bir çatıya topla
    if 'admin' in x_lower:
        return 'Admin'
    # Mobil kelimesi, app, ios veya android geçenleri Mobil yap
    elif any(kelime in x_lower for kelime in ['mobil', 'app', 'ios', 'android']):
        return 'Mobil'
    # Geri kalan her şeyi Masaüstü kabul et
    else:
        return 'Masaüstü'

df['Platform_Gruplu'] = df['Platform'].apply(platform_birlestir)

print("--- Platform Dağılımı ---")
print(df['Platform_Gruplu'].value_counts())
print("\n")

# Özellik seçimi listesini güncelleyelim ('Platform' yerine 'Platform_Gruplu' geldi)
secilen_ozellikler = ['Platform_Gruplu', 'Kaynak', 'Ödeme Tipi', 'Ana Kategori', 'AltKat_Gruplu', 'İl_Gruplu']

X = df[secilen_ozellikler].fillna('Bilinmiyor')
y = df['target']

# drop_first=True parametresi "Dummy Variable Trap" sorununu önler
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"BÜYÜK BAŞARI: Matris sütun sayımız 146'dan {X_encoded.shape[1]} sütuna düştü!")
display(X_encoded.head(3))

--- Hedef Değişken (0: Teslim, 1: İade, 2: İptal) Dağılımı ---
target
0    81.190926
1    10.302457
2     8.506616
Name: proportion, dtype: float64


--- İl Dağılımı (Gruplama Sonrası) ---
İl_Gruplu
Diğer       876
İstanbul    831
Ankara      407
İzmir       355
Antalya     147
Name: count, dtype: int64


--- Platform Dağılımı ---
Platform_Gruplu
Mobil       2500
Masaüstü     640
Admin         34
Name: count, dtype: int64


BÜYÜK BAŞARI: Matris sütun sayımız 146'dan 43 sütuna düştü!


,Platform_Gruplu_Masaüstü,Platform_Gruplu_Mobil,Kaynak_Enhencer,Kaynak_IGShopping,Kaynak_chatgpt.com,Kaynak_corporatebenefit,Kaynak_facebook,Kaynak_fb,Kaynak_ig,Kaynak_personaclick,...,İl_Gruplu_Ankara,İl_Gruplu_Antalya,İl_Gruplu_Aydın,İl_Gruplu_Balıkesir,İl_Gruplu_Bursa,İl_Gruplu_Diğer,İl_Gruplu_Kocaeli,İl_Gruplu_Muğla,İl_Gruplu_İstanbul,İl_Gruplu_İzmir
0,False,True,False,False,False,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,False,True,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3174 entries, 0 to 3173
Data columns (total 36 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Sipariş No       3174 non-null   str    
 1   Üye Grup Kodu    2329 non-null   str    
 2   Üye Grubu        2329 non-null   str    
 3   İl (Teslimat)    3174 non-null   str    
 4   İlçe (Teslimat)  3174 non-null   str    
 5   Ülke (Teslimat)  3174 non-null   str    
 6   Döviz Cinsi      3174 non-null   str    
 7   Sistem Kuru      3174 non-null   str    
 8   Döviz Tutar      3174 non-null   float64
 9   Tutar            3173 non-null   float64
 10  KDV              3168 non-null   float64
 11  Kargo Toplamı    748 non-null    float64
 12  Hizmet Bedeli    543 non-null    float64
 13  Kargo            3174 non-null   str    
 14  Ödeme Tipi       3173 non-null   str    
 15  Alt Ödeme Tipi   215 non-null    str    
 16  Banka            2957 non-null   str    
 17  Kart             2827 non